# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# Week 37 lesson — Two dice and conditional distributions

Used in the exercise session on 11 September 2026.

Roll a fair six-sided die twice, independently. Let $X$ be the **sum** of the
two rolls and $Y$ the **number of sixes** rolled. From the 36 equally likely
outcomes, we calculate a joint probability mass function (pmf), a marginal
pmf, a conditional pmf, and a conditional expectation. We also check whether
$X$ and $Y$ are independent.

Run the cells in order. The final exercises use the tower property to connect
conditional and unconditional expectations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb

rng = np.random.default_rng(2026)

## The model: two independent rolls of a fair die

Let $\Omega=\{1,\dots,6\}^2$. Every subset of $\Omega$ is an event, and each
ordered pair $\omega=(\omega_1,\omega_2)$ has probability
$\mathbb P(\{\omega\})=1/36$. Define

$$
X(\omega)=\omega_1+\omega_2 \quad\text{(the sum)}, \qquad
Y(\omega)=\mathbf 1_{\{\omega_1=6\}}+\mathbf 1_{\{\omega_2=6\}}
\quad\text{(the number of sixes)}.
$$

Each indicator is 1 when the corresponding roll is a six and 0 otherwise.
Thus $X$ takes values in $\{2,\dots,12\}$ and $Y$ takes values in $\{0,1,2\}$.

In [ ]:
die_values = np.arange(1, 7)

outcomes = []

for roll1 in die_values:
    for roll2 in die_values:
        outcomes.append([roll1, roll2])

outcomes = np.array(outcomes)
outcome_probabilities = np.full(len(outcomes), 1 / 36)

x_of_outcome = outcomes.sum(axis=1)                # the sum of both rolls
y_of_outcome = (outcomes == 6).sum(axis=1)          # the number of sixes

x_values = np.arange(2, 13)
y_values = np.array([0, 1, 2])

print(f"{len(outcomes)} equally likely outcomes, each with probability: {outcome_probabilities[0]:.4f}")
print("First few outcomes [roll1, roll2]: X, Y")
for outcome, x, y in list(zip(outcomes, x_of_outcome, y_of_outcome))[:6]:
    print(f" {(outcome)}: X={x}, Y={y}")

## The joint pmf $f_{X,Y}$

Every part below is read off from the joint pmf
$f_{X,Y}(x,y)=\mathbb P(X=x,Y=y)$, obtained by summing the probability of every
outcome with that $(x,y)$ pair.

In [ ]:
joint_pmf = np.zeros((len(x_values), len(y_values)))
for i, x in enumerate(x_values):
    for j, y in enumerate(y_values):
        matching_outcomes = (x_of_outcome == x) & (y_of_outcome == y)
        joint_pmf[i, j] = outcome_probabilities[matching_outcomes].sum()

print("f_{X,Y}(x,y)   y=0     y=1     y=2")
for x, row in zip(x_values, joint_pmf):
    print(f"x={x:2d}          " + "  ".join(f"{value:.4f}" for value in row))
print(f"\nTotal mass: {joint_pmf.sum():.4f}")

## a) The pmf of $Y$

$Y$ counts sixes in two independent rolls, each landing on six with
probability $1/6$, so $Y\sim\mathrm{Binomial}(2,1/6)$:

$$
f_Y(y)=\binom{2}{y}\Big(\frac16\Big)^{y}\Big(\frac56\Big)^{2-y}, \qquad y=0,1,2.
$$

We can also get $f_Y$ directly as the marginal of the joint table, by summing
$f_{X,Y}(x,y)$ over $x$.

In [ ]:
f_Y = joint_pmf.sum(axis=0)   # marginal of Y: sum the joint pmf over x

p_six = 1 / 6
f_Y_binom = np.array([comb(2, y) * p_six**y * (1 - p_six)**(2 - y) for y in y_values])

print("y   f_Y(y) [from joint table]   f_Y(y) [Binomial(2,1/6) formula]")
for y, fy, fyf in zip(y_values, f_Y, f_Y_binom):
    print(f"{y}   {fy:.4f}                     {fyf:.4f}")

## b) The joint probabilities $f_{X,Y}(x,1)$

Exactly one six means one die shows $6$ and the other shows some
$k\in\{1,\dots,5\}$. For each $x=6+k$, there are two ordered outcomes,
$(6,k)$ and $(k,6)$, each with probability $1/36$. Therefore,

$$
f_{X,Y}(x,1)=
\begin{cases}
\dfrac{2}{36}=\dfrac{1}{18}, & x\in\{7,8,9,10,11\},\\
0, & \text{otherwise.}
\end{cases}
$$

These are the entries in column $y=1$ of the joint table. They are joint
probabilities; we have not yet conditioned on $Y=1$.

In [ ]:
y1_index = np.flatnonzero(y_values == 1)[0]
f_XY_at_1 = joint_pmf[:, y1_index]

print("x    f_{X,Y}(x,1)")
for x, value in zip(x_values, f_XY_at_1):
    print(f"{x:2d}   {value:.4f}")

## c) Summing the joint probabilities at $Y=1$

Summing over $x$ gives the marginal probability from part (a):

$$
\sum_{x=2}^{12} f_{X,Y}(x,1)=f_Y(1)=5\cdot\frac{1}{18}=\frac{5}{18}.
$$

In [ ]:
p_Y1 = f_Y[y1_index]

print(f"Sum of f_XY(x,1): {f_XY_at_1.sum():.4f}")
print(f"f_Y(1):          {p_Y1:.4f}")

## d) The conditional probability $f_{X\mid Y}(7\mid 1)$

For any $y$ with $f_Y(y)>0$, the conditional pmf is

$$
f_{X\mid Y}(x\mid y)=\frac{f_{X,Y}(x,y)}{f_Y(y)}.
$$

Here $f_Y(1)=5/18>0$, so

$$
f_{X\mid Y}(7\mid 1)=\frac{1/18}{5/18}=\frac15.
$$

The same calculation applies to $x=8,9,10,11$. Thus, conditional on $Y=1$,
$X$ is uniform on $\{7,8,9,10,11\}$.

In [ ]:
f_X_given_Y1 = f_XY_at_1 / p_Y1

print("x    f_{X|Y}(x|1)")
for x, value in zip(x_values, f_X_given_Y1):
    print(f"{x:2d}   {value:.4f}")

value_at_7 = f_X_given_Y1[x_values == 7][0]
print(f"\nf_X|Y(7|1) = {value_at_7:.4f}")
print(f"Total conditional mass: {f_X_given_Y1.sum():.4f}")

## e) Are $X$ and $Y$ independent?

Independence requires $f_{X,Y}(x,y)=f_X(x)f_Y(y)$ for every pair $(x,y)$.
Equivalently, $f_{X\mid Y}(x\mid y)=f_X(x)$ for every $x$ and every $y$
with $f_Y(y)>0$.

One counterexample is enough to disprove independence. Here $X=2$ has
probability $1/36$, but is impossible when $Y=1$. Since $f_Y(1)>0$,
$X$ and $Y$ are not independent.

In [ ]:
f_X = joint_pmf.sum(axis=1)  # marginal of X: sum over y

p_X2 = f_X[x_values == 2][0]
joint_at_2_1 = f_XY_at_1[x_values == 2][0]
product_at_2_1 = p_X2 * p_Y1

print(f"f_XY(2,1)      = {joint_at_2_1:.4f}")
print(f"f_X(2)*f_Y(1)  = {product_at_2_1:.4f}")
print(f"f_X(2)         = {p_X2:.4f}")
print(f"f_X|Y(2|1)     = {f_X_given_Y1[x_values == 2][0]:.4f}")
print("\nThe joint probability differs from the product, so X and Y are not independent.")

In [ ]:
width = 0.4
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(x_values - width / 2, f_X, width=width, label="marginal $f_X(x)$")
ax.bar(x_values + width / 2, f_X_given_Y1, width=width, label="conditional $f_{X|Y}(x|1)$")
ax.set(xlabel="x", ylabel="probability", title="Marginal vs. conditional law of X")
ax.set_xticks(x_values)
ax.legend()
plt.tight_layout()
plt.show()

## f) The conditional expectation $\mathbb E[X\mid Y=1]$

Using the conditional pmf from part (d),

$$
\mathbb E[X\mid Y=1]
=\sum_{x=2}^{12} x\,f_{X\mid Y}(x\mid 1)
=\frac{7+8+9+10+11}{5}=9.
$$

We can also use the joint probabilities:

$$
\mathbb E[X\mid Y=1]
=\frac{\sum_{x=2}^{12} x\,f_{X,Y}(x,1)}{f_Y(1)}.
$$

The numerator is a **weighted** sum, unlike the unweighted sum in part (c).
Below we compare both exact calculations with a simulated average over
independent pairs of fair-die rolls for which $Y=1$. The simulated value
is an approximation.

In [ ]:
weighted_sum = np.sum(x_values * f_XY_at_1)

E_X_given_Y1_formula = weighted_sum / p_Y1
E_X_given_Y1_direct = np.sum(x_values * f_X_given_Y1)

n_sim = 200_000
rolls = rng.integers(1, 7, size=(n_sim, 2))
X_sim = rolls.sum(axis=1)
Y_sim = (rolls == 6).sum(axis=1)
selected_sums = X_sim[Y_sim == 1]

print(f"E[X|Y=1] from the weighted joint sum: {E_X_given_Y1_formula:.4f}")
print(f"E[X|Y=1] from the conditional pmf:    {E_X_given_Y1_direct:.4f}")
print(f"Retained simulated pairs: {selected_sums.size}")
if selected_sums.size > 0:
    print(f"E[X|Y=1] estimated by simulation:    {selected_sums.mean():.4f}")
else:
    print("No pairs with Y=1 were sampled; increase n_sim.")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(x_values, f_X_given_Y1, color="steelblue", alpha=0.85,
       label=r"$f_{X\mid Y}(x\mid 1)$")
ax.axvline(E_X_given_Y1_formula, color="black", linestyle="--",
           label=fr"$\mathbb{{E}}[X\mid Y=1]={E_X_given_Y1_formula:.0f}$")
ax.set(xlabel="x", ylabel="conditional probability",
       title="Conditional distribution of X given Y=1")
ax.set_xticks(x_values)
ax.legend()
plt.tight_layout()
plt.show()

## Try it yourself: the tower property

In this finite model, $\mathbb E[X\mid Y]$ is the random variable whose value
is $\mathbb E[X\mid Y=y]$ whenever $Y=y$. The tower property states that

$$
\mathbb E[X]=\mathbb E\big[\mathbb E[X\mid Y]\big]
=\sum_{y=0}^{2} f_Y(y)\,\mathbb E[X\mid Y=y].
$$

1. **Check the tower property for the two dice.** Compute
   $\mathbb E[X\mid Y=0]$ and $\mathbb E[X\mid Y=2]$ as in part (f).
   Use them in the weighted sum above and compare with $\mathbb E[X]$
   computed directly from $f_X$.

2. **A random number of coin flips.** Roll a fair die once to get
   $N\in\{1,\dots,6\}$. Take a sequence of independent fair-coin flips,
   independent of the die, and let $H$ count the heads in its first $N$ flips.
   Given $N=n$, $H$ has the $\mathrm{Binomial}(n,1/2)$ distribution, so
   $\mathbb E[H\mid N=n]=n/2$. Use the tower property to find $\mathbb E[H]$.
   Then find the pmf of $H$ by conditioning on $N$:
   $$
   f_H(h)=\sum_{n=1}^{6}\frac16\,f_{H\mid N}(h\mid n),
   \qquad h=0,\dots,6.
   $$
   Remember that $f_{H\mid N}(h\mid n)=0$ when $h>n$.